In [ ]:
# Average mood scores by genre
if len(df) > 1 and df["genre"].nunique() > 1:
    genre_moods = df.groupby("genre")[["happy", "aggressive", "relaxed", "sad", "danceability"]].mean()
    
    fig, ax = plt.subplots(figsize=(12, max(4, len(genre_moods) * 0.5)))
    genre_moods.plot(kind="barh", ax=ax)
    ax.set_xlabel("Average Score")
    ax.set_title("Average Mood/Danceability by DJ Genre")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
    
    print("\nMood averages per genre:")
    display(genre_moods.round(3))
else:
    print("Need tracks across multiple genres for this analysis.")

## 4. Mood by Genre

Check for systematic biases: does the model always classify certain genres with the same mood?

In [ ]:
# Correlation matrix
corr_cols = ["bpm", "energy_mean", "happy", "aggressive", "relaxed", "sad", "danceability"]
if len(df) > 2:
    corr = df[corr_cols].corr()
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax,
                vmin=-1, vmax=1)
    ax.set_title("Correlation: Audio Features vs Mood/Danceability")
    plt.tight_layout()
    plt.show()
else:
    print("Need more tracks for meaningful correlations.")

## 3. Correlation Analysis

How do mood/danceability correlate with BPM, energy, and each other?

In [ ]:
from math import pi

mood_cols = ["happy", "aggressive", "relaxed", "sad", "danceability"]
n_moods = len(mood_cols)
angles = [n * 2 * pi / n_moods for n in range(n_moods)] + [0]  # close the plot

n_tracks = len(df)
cols = min(4, n_tracks)
rows_needed = max(1, (n_tracks + cols - 1) // cols)
fig, axes = plt.subplots(rows_needed, cols, figsize=(4 * cols, 4 * rows_needed), 
                          subplot_kw=dict(polar=True))
if n_tracks == 1:
    axes = np.array([axes])
axes = np.atleast_2d(axes)

for idx, (_, row) in enumerate(df.iterrows()):
    ax = axes[idx // cols][idx % cols]
    values = [row[c] for c in mood_cols] + [row[mood_cols[0]]]
    ax.plot(angles, values, linewidth=2)
    ax.fill(angles, values, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(mood_cols, fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title(f"{row['title'][:20]}\n[{row['genre']}]", fontsize=9, pad=15)

# Hide unused subplots
for idx in range(n_tracks, rows_needed * cols):
    axes[idx // cols][idx % cols].set_visible(False)

plt.tight_layout()
plt.show()

## 2. Mood Radar Charts

Visualize mood profile for each track as a radar/spider chart.

In [ ]:
# Build mood dataframe
rows = []
for t in tracks_with_clf:
    clf = t.classification
    profile = map_to_dj_genres(clf.genres)
    rows.append({
        "title": t.title,
        "genre": profile.primary_genre,
        "bpm": t.bpm,
        "energy_mean": np.mean(t.energy_curve),
        "happy": clf.mood_happy,
        "aggressive": clf.mood_aggressive,
        "relaxed": clf.mood_relaxed,
        "sad": clf.mood_sad,
        "danceability": clf.danceability,
        "voice_instrumental": clf.voice_instrumental,
    })

df = pd.DataFrame(rows)
display(df.style.format({
    "bpm": "{:.1f}", "energy_mean": "{:.2f}",
    "happy": "{:.3f}", "aggressive": "{:.3f}", "relaxed": "{:.3f}",
    "sad": "{:.3f}", "danceability": "{:.3f}", "voice_instrumental": "{:.3f}",
}).background_gradient(subset=["happy", "aggressive", "relaxed", "sad", "danceability"], cmap="YlOrRd"))

## 1. Mood & Danceability Overview

Per-track mood scores and danceability. Check if values seem reasonable.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from agent_dj.analyzer.track_store import TrackStore
from agent_dj.analyzer.genre_taxonomy import map_to_dj_genres

sns.set_theme(style="whitegrid")

store = TrackStore("../tracks.db")
tracks = store.get_all()
tracks_with_clf = [t for t in tracks if t.classification]
print(f"Tracks with classification: {len(tracks_with_clf)} / {len(tracks)}")

# Experiment 02: Mood & Danceability Evaluation

Evaluate mood classifiers (happy, aggressive, relaxed, sad) and danceability scores.

**Goals:**
- Do mood scores make sense for our sample tracks?
- Are there systematic biases (e.g., all jazz = relaxed)?
- Is danceability useful for set planning?
- How do mood/danceability correlate with audio features (BPM, energy)?